# Lights, Camera, Animation

In [[A0635038-E981-4928-AB28-B1A2C883E114|Creating a 3D Scene]] you built worlds that stood perfectly still. This notebook brings them to life: shapes that spin, slide, and bounce, groups of shapes that move as one, lights that set a mood, and a camera that follows the action.

An animated scene is drawn again and again, many times every second. Each drawing is called a **frame**. If a shape is in a slightly different place in each frame, your eyes see it moving, exactly like a flip book.

Run the cell below. When you have seen enough, press the **Stop** button (■) on the cell.

> **Important:** An animation keeps running until you stop it. Press **Stop** before you run a different cell.

In [ ]:
from codetto import graphics3d

scene = graphics3d.Scene()
scene.set_sky('#1a1a2e')
scene.set_ground(length=10, width=10)

box = graphics3d.Shapes.Box(width=2, height=2, depth=2)
box.set_position(0, 1, 0)
box.set_material(graphics3d.Material.Tiles.Checkerboard)
scene.add(box)

angle = 0

@scene.on_frame
def animate(dt):
  global angle
  angle += 90 * dt
  box.set_rotation(y=angle)

scene.run()

# How Animation Works

Three new pieces made that box spin.

**`@scene.on_frame`** goes on the line above a function. It tells the scene: "call this function just before you draw each frame." Whatever the function changes shows up in the next frame.

**`dt`** is handed to your function every time. It is the number of seconds since the previous frame, usually a tiny number like `0.016`. Multiplying a speed by `dt` turns "per second" into "per frame":

```python
angle += 90 * dt    # 90 degrees every second
```

Fast computers draw more frames than slow ones. Because every frame only adds its own small slice of time, the box turns at 90 degrees per second on all of them.

**`scene.run()`** starts the animation and keeps the cell running so the frames keep coming. Code written after `scene.run()` waits until you press Stop.

One more detail: `angle` is created outside the function, and the function changes it. The line `global angle` is what lets it do that. Without it, Python would report an error on the `angle += ...` line.

# Back and Forth

Adding to a number every frame makes something travel in one direction forever. To make a shape go back and forth, use `math.sin`.

Give `math.sin` a number that keeps growing, such as the time since the animation started, and it gives back a number that glides smoothly between `-1` and `1`, over and over. Multiply that by a distance and you have a position that sways from side to side.

```python
x = math.sin(t * SPEED) * DISTANCE
```

Here `t` counts the seconds since the start (it adds up every `dt`). `SPEED` sets how quickly the ball sways and `DISTANCE` sets how far it goes. Change the sliders and run the cell again.

In [ ]:
import math

SPEED = 2 #@param {type:"slider", min:0.5, max:6, step:0.5}
DISTANCE = 4 #@param {type:"slider", min:1, max:6, step:1}

scene = graphics3d.Scene()
scene.set_sky('#0f3460')
scene.set_ground(length=8, width=14)

ball = graphics3d.Shapes.Sphere(diameter=1)
ball.set_position(0, 0.5, 0)
ball.set_material(graphics3d.Material.Sports.Tennis)
scene.add(ball)

ring = graphics3d.Shapes.Torus(diameter=1.6, thickness=0.2)
ring.set_color('#f5a623')
scene.add(ring)

t = 0

@scene.on_frame
def animate(dt):
  global t
  t += dt
  ball.set_position(math.sin(t * SPEED) * DISTANCE, 0.5, 0)
  ring.set_position(0, 2.5 + math.sin(t * SPEED * 2) * 0.5, 0)

scene.run()

# Asking a Shape Where It Is

Every `set_` command has a matching `get_` command that reads the value back:

- `shape.get_position()` gives back three numbers: `x`, `y`, and `z`.
- `shape.get_rotation()` and `shape.get_scale()` work the same way.
- `shape.get_color()` gives back the hex code.

You can unpack the three numbers into three variables on one line:

```python
x, y, z = ball.get_position()
```

This is handy in an animation: read where the shape is now, change it a little, and put it back. The ball below does exactly that. Each frame it reads its height, falls a little faster than the frame before, and bounces when it reaches the ground, losing a quarter of its speed on every bounce.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#0f3460')
scene.set_ground(length=10, width=10)

ball = graphics3d.Shapes.Sphere(diameter=1)
ball.set_position(0, 6, 0)
ball.set_color('#e94560')
scene.add(ball)

speed = 0

@scene.on_frame
def animate(dt):
  global speed
  speed -= 9.8 * dt

  x, y, z = ball.get_position()
  y += speed * dt

  if y < 0.5:
    y = 0.5
    speed = abs(speed) * 0.75

  ball.set_position(x, y, z)

scene.run()

# Groups

Most interesting objects are built from several shapes. A car is a body, a cabin, and four wheels. Moving it one shape at a time would take six `set_position` calls every frame, and one mistake would leave a wheel behind.

A **group** glues shapes together so they act as one object:

```python
car = graphics3d.Group()
car.add(body)        # add shapes to the group, not to the scene
car.add(cabin)
scene.add(car)       # then add the whole group to the scene
```

Positions of shapes inside a group are measured from the **group's** center, not the world's. After that, `car.set_position`, `car.set_rotation`, and `car.set_scale` move, turn, and resize everything in it at once.

In the cell below, a single `car.set_position` call drives all six shapes across the ground.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky('#87ceeb')
ground = scene.set_ground(length=24, width=24)
ground.set_material(graphics3d.Material.Gravel.DarkGray)
ground.set_tiling(8)

car = graphics3d.Group()

body = graphics3d.Shapes.Box(width=3, height=0.6, depth=1.4)
body.set_position(0, 0.65, 0)
body.set_color('#e94560')
car.add(body)

cabin = graphics3d.Shapes.Box(width=1.5, height=0.6, depth=1.2)
cabin.set_position(-0.2, 1.25, 0)
cabin.set_color('#a8dadc')
car.add(cabin)

for x in [-1, 1]:
  for z in [-0.7, 0.7]:
    wheel = graphics3d.Shapes.Cylinder(diameter=0.7, height=0.3)
    wheel.set_position(x, 0.35, z)
    wheel.set_rotation(x=90)
    wheel.set_color('#222222')
    car.add(wheel)

scene.add(car)

@scene.on_frame
def animate(dt):
  x, y, z = car.get_position()
  x += 4 * dt
  if x > 10:
    x = -10
  car.set_position(x, y, z)

scene.run()

# Orbiting with a Group

Groups have a second trick. A group turns around its **own center**, and the shapes inside it are carried along, like a ball on the end of a string.

So to make a moon circle a planet:

1. Make a group and leave it at the center of the world, where the planet is.
2. Put the moon in the group, 4 units out to the side.
3. Rotate the **group**. The moon swings around in a circle.

No `math.sin` needed. The planet below is not in the group, so it can spin at its own speed.

In [ ]:
scene = graphics3d.Scene()
scene.set_sky(graphics3d.Sky.DEEP_SPACE)

planet = graphics3d.Shapes.Sphere(diameter=3, segments=32)
planet.set_material(graphics3d.Material.Planets.Earth)
scene.add(planet)

orbit = graphics3d.Group()

moon = graphics3d.Shapes.Sphere(diameter=0.8, segments=24)
moon.set_position(4, 0, 0)
moon.set_material(graphics3d.Material.Planets.Moon)
orbit.add(moon)

scene.add(orbit)

t = 0

@scene.on_frame
def animate(dt):
  global t
  t += dt
  planet.set_rotation(y=t * 20)
  orbit.set_rotation(y=t * 60)

scene.run()

# Setting the Mood with Light

Every scene starts with one light already switched on: the **ambient light**. It has no position. It lights everything evenly from all directions, like an overcast day. You reach it through `scene.ambient`.

- `scene.ambient.set_brightness(value)` goes from `0` (pitch black) to `100` (fully lit). A new scene starts at `90`.
- `scene.ambient.set_color(color)` tints everything the light touches.

Those two commands change the whole feeling of a scene without touching a single shape. Pick a mood from the dropdown and run the cell again.

In [ ]:
MOOD = "sunset" #@param ["daylight", "sunset", "moonlight", "cave"]

moods = {
  'daylight': ('#ffffff', 90),
  'sunset': ('#ff9955', 55),
  'moonlight': ('#aabbff', 25),
  'cave': ('#445533', 15),
}
color, brightness = moods[MOOD]

scene = graphics3d.Scene()
scene.set_sky('#1a1a2e')
scene.set_ground(length=12, width=12)

for x, shape_color in [(-3, '#e94560'), (0, '#ffffff'), (3, '#44cc88')]:
  ball = graphics3d.Shapes.Sphere(diameter=1.6, segments=24)
  ball.set_position(x, 0.8, 0)
  ball.set_color(shape_color)
  scene.add(ball)

scene.ambient.set_color(color)
scene.ambient.set_brightness(brightness)

print(f'{MOOD}: color {color}, brightness {brightness}')

# Adding Lights

A **point light** is like a bare light bulb. It sits at one spot and shines outward in every direction, so shapes are bright on the side facing it and dark on the far side.

```python
lamp = scene.add_light(0, 4, 0)    # x, y, z of the bulb
lamp.set_color('#ff8844')
lamp.set_brightness(100)           # 0 to 100
lamp.set_visible(True)             # show a small glowing ball where the light is
```

Two tips:

- Turn the ambient light **down** first. In a fully lit scene an extra light is hard to notice.
- A light is invisible unless you call `set_visible(True)`. Switch it on while you are working out where the light should go.

> **Tip:** Point lights are much stronger on plain-colored shapes than on shapes with a material. If a shape near a light turns pure white, lower the light's brightness or give the shape a material.

Lights can be animated like shapes. This one flies in a circle: `math.sin` for `x` and `math.cos` for `z` trace out a circle between them.

In [ ]:
import math

LIGHT_COLOR = "#ff8844" #@param ["#ffffff", "#ff8844", "#4488ff", "#44ff88", "#ff3355"]

scene = graphics3d.Scene()
scene.set_sky('#1a1a2e')
ground = scene.set_ground(length=14, width=14)
ground.set_material(graphics3d.Material.Gravel.DarkGray)
ground.set_tiling(4)

scene.ambient.set_brightness(10)

for x, z in [(-3, 0), (0, 0), (3, 0), (-1.5, 3), (1.5, 3)]:
  block = graphics3d.Shapes.Box(width=1.2, height=1.2, depth=1.2)
  block.set_position(x, 0.6, z)
  block.set_material(graphics3d.Material.Marble.Gray)
  scene.add(block)

lamp = scene.add_light(0, 4, 0)
lamp.set_color(LIGHT_COLOR)
lamp.set_visible(True)

t = 0

@scene.on_frame
def animate(dt):
  global t
  t += dt
  lamp.set_position(math.sin(t) * 5, 4, math.cos(t) * 4)

scene.run()

# A Camera That Follows

When something travels a long way, a camera that stays put soon loses sight of it. `scene.camera.follow(shape)` keeps the camera pointed at a shape, frame after frame, wherever it goes.

```python
scene.camera.follow(ball)                # keep the ball in the middle of the view
scene.camera.follow(ball, distance=8)    # ...from 8 units away
scene.camera.follow(None)                # stop following
```

You can follow a group too, and you can still drag to look at the shape from a different side while the camera follows it.

The ball below rolls in a big figure of eight. Run the cell with **FOLLOW** ticked, then untick it and run again to compare.

In [ ]:
import math

FOLLOW = True #@param {type:"boolean"}

scene = graphics3d.Scene()
scene.set_sky(graphics3d.Sky.PURE_SKY)
ground = scene.set_ground(length=30, width=30)
ground.set_material(graphics3d.Material.Grass.Bright)
ground.set_tiling(12)

ball = graphics3d.Shapes.Sphere(diameter=1, segments=24)
ball.set_position(0, 0.5, 0)
ball.set_material(graphics3d.Material.Sports.Soccerball)
scene.add(ball)

for x in [-10, -5, 0, 5, 10]:
  post = graphics3d.Shapes.Cylinder(diameter=0.4, height=2)
  post.set_position(x, 1, 7)
  post.set_color('#ffffff')
  scene.add(post)

scene.camera.set_position(0, 4, -12)
if FOLLOW:
  scene.camera.follow(ball, distance=10)

t = 0

@scene.on_frame
def animate(dt):
  global t
  t += dt
  ball.set_position(math.sin(t * 0.6) * 10, 0.5, math.sin(t * 1.2) * 5)

scene.run()

# Putting It Together: A Campfire at Night

This last scene combines lighting, animation, and camera work.

- **Lighting:** a dim, blue ambient light for the night, and an orange point light inside the fire.
- **Animation:** the fire's brightness wobbles using two `math.sin` waves at different speeds, which looks like a flicker.
- **A moving light:** a firefly (a small visible light) circles the camp, bobbing up and down as it goes.
- **Camera:** with **ORBIT** ticked, the camera is moved a little every frame so it slowly circles the fire. Untick it to steer the camera yourself.

Ideas to try:

- Add more trees to the list, or a tent made from a four-sided cone (`tessellation=4`).
- Give the camp a second firefly that circles the other way.
- Change the fire to a magical color and match the ambient light to it.
- Make the camera rise and fall as it circles.

Next, make your scenes respond to clicks and key presses in [[80FE9F9A-F226-4BF4-AC1B-D5A4B4F7647E|Interactive 3D]].

In [ ]:
import math

ORBIT = True #@param {type:"boolean"}

scene = graphics3d.Scene()
scene.set_sky('#0b1026')
ground = scene.set_ground(length=24, width=24)
ground.set_material(graphics3d.Material.Grass.Dark)
ground.set_tiling(8)

scene.ambient.set_color('#8899ff')
scene.ambient.set_brightness(20)

# The fire: three crossed logs with a glowing light in the middle
for turn in [0, 60, 120]:
  log = graphics3d.Shapes.Cylinder(diameter=0.3, height=2)
  log.set_position(0, 0.15, 0)
  log.set_rotation(x=90, y=turn)
  log.set_material(graphics3d.Material.Wood.Oak)
  scene.add(log)

fire = scene.add_light(0, 0.6, 0)
fire.set_color('#ff8844')
fire.set_visible(True)

# Trees around the camp
for x, z in [(-5, 3), (-3, -5), (4, 5), (6, -2), (0, 7)]:
  trunk = graphics3d.Shapes.Cylinder(diameter=0.4, height=1.2)
  trunk.set_position(x, 0.6, z)
  trunk.set_material(graphics3d.Material.Wood.Oak)
  scene.add(trunk)

  leaves = graphics3d.Shapes.Cone(diameter=2.2, height=3.5)
  leaves.set_position(x, 2.9, z)
  leaves.set_material(graphics3d.Material.Grass.Dark)
  scene.add(leaves)

# A firefly: a small visible light that circles the camp
firefly = scene.add_light(3.5, 1.5, 0)
firefly.set_color('#ccff66')
firefly.set_brightness(15)
firefly.set_visible(True)

scene.camera.look_at(0, 1, 0)

t = 0

@scene.on_frame
def animate(dt):
  global t
  t += dt

  flicker = math.sin(t * 9) * 0.5 + math.sin(t * 23) * 0.5
  fire.set_brightness(75 + flicker * 20)
  fire.set_position(0, 0.6 + flicker * 0.05, 0)

  firefly.set_position(math.sin(t * 0.8) * 3.5, 1.5 + math.sin(t * 3) * 0.4, math.cos(t * 0.8) * 3.5)

  if ORBIT:
    scene.camera.set_position(math.sin(t * 0.2) * 11, 4, -math.cos(t * 0.2) * 11)

scene.run()

# Check Your Understanding